In [1]:
2+1

3

In [5]:
from research_analysis_generation.logger.custom_logger import CustomLogger

logger = CustomLogger().get_logger()
logger.info("upload")

{"timestamp": "2026-04-29T07:05:01.609594Z", "level": "info", "event": "upload"}


In [2]:
from typing import TypedDict, List
from pydantic import Field, BaseModel

In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [4]:
class Analyst(BaseModel):
    name: str = Field(description="name of the analyst")
    role: str = Field(description="role of the analyst")

In [9]:
class Perspective(BaseModel):
    analysts: List[Analyst] = Field(description="list of analysts")

In [8]:
class GenerateState(TypedDict):
    topic: str
    max_analyst: int 
    human_feedback: str 
    analysts: List[Analyst] 

In [7]:
analyst_instructions="""You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:

1. First, review the research topic:
{topic}
        
2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: 
        
{human_feedback}
    
3. Determine the most interesting themes based upon documents and / or feedback above.
                    
4. Pick the top {max_analyst} themes.

5. Assign one analyst to each theme."""

In [10]:
from research_analysis_generation.utils.model_loader import ModelLoader

model_loader = ModelLoader()
llm = model_loader.load_llm()

{"timestamp": "2026-04-29T16:22:43.992021Z", "level": "info", "event": "Initializing ApiKeyManager"}
{"timestamp": "2026-04-29T16:22:43.993504Z", "level": "info", "event": "OPENAI_API_KEY loaded successfully from environment"}
{"timestamp": "2026-04-29T16:22:43.994036Z", "level": "info", "event": "GROQ_API_KEY loaded successfully from environment"}
{"config_keys": ["embedding_model", "llm", "retriever", "chunking"], "timestamp": "2026-04-29T16:22:43.999409Z", "level": "info", "event": "YAML configuration loaded successfully"}
{"provider": "openai", "model": "gpt-4o-mini", "timestamp": "2026-04-29T16:22:44.000701Z", "level": "info", "event": "Loading LLM"}
{"provider": "openai", "model": "gpt-4o-mini", "timestamp": "2026-04-29T16:22:44.003590Z", "level": "info", "event": "LLM loaded successfully"}


In [6]:
llm.invoke("hi").content

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'Hello! How can I assist you today?'

In [17]:
def create_analyst(state:GenerateState):
    topic = state.get("topic")
    max_analyst = state.get("max_analyst")
    human_feedback = state.get("human_feedback")
    
    structured_llm = llm.with_structured_output(Perspective)
      
    system_message = analyst_instructions.format(
        topic = topic,
        max_analyst = max_analyst,
        human_feedback = human_feedback
    )
    
    analysts = structured_llm.invoke([SystemMessage(content=system_message)] + [HumanMessage(content="Generate the set of analysts.")])
    
    return {"analysts": analysts.analysts}

In [19]:
create_analyst(
    {
        'topic': 'health',
        'max_analysts': 1,
        'human_analyst_feedback': 'give the real info'}
    )


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'analysts': [Analyst(name='Dr. Emily Carter', role='Public Health Analyst'),
  Analyst(name='James Thompson', role='Healthcare Data Scientist'),
  Analyst(name='Sarah Patel', role='Mental Health Researcher'),
  Analyst(name='Dr. Michael Chen', role='Epidemiologist'),
  Analyst(name='Laura Gomez', role='Health Policy Analyst')]}